In [31]:
!pip install biopython

Defaulting to user installation because normal site-packages is not writeable


# Pre-processing (always need to run)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import csv
import scipy
import string
from scipy.stats import spearmanr, pearsonr
import numpy as np
from Bio import SeqIO

import sys
import argparse
import re 
import warnings
import pandas as pd
import numpy as np
import duckdb
conn = duckdb.connect('/global/scratch/projects/fc_mvslab/OpenProjects/EChase/TREBLEseq_ismaybethenewcibername/A10_sequencing/v2/analysis2.db') # connect to database with TBB map

In [3]:
test_4_10_iii_path = '/global/scratch/projects/fc_mvslab/data/sequencing/20240626_MZ_Spike_EC-CiBER/results/assembled_reads/RPTR_reads/Staller_RPTR_4_10_MVS_0094_I1_TGACGTCGTC_GATAAGTACG_S436.fastq.gz.assembled.fastq'
test_4_10_ii_path = '/global/scratch/projects/fc_mvslab/data/sequencing/CZB_Apr2024/20240425/EC_Ciber2/results/assembled_reads/MINI_RPTR_4_10_S78.fastq.gz.assembled.fastq'
full_4_10_iii_path = '/global/scratch/projects/fc_mvslab/data/sequencing/CZB_Sep2024/ciber2_iii_MZ001/EC_Ciber2_iii_Gcn4/results/assembled/RPTR/RPTR_4_10_S38_L001.fastq.gz.assembled.fastq'

In [4]:
def complement(seq):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A', 'N':'N', 'X':'X'} 
    bases = list(seq) 
    bases = [complement[base] for base in bases] 
    return ''.join(bases)
def reverse_complement(s):
        return complement(s[::-1])
    
    
def getmid(seq, pre, post, bclen):
    # seq = the sequence to parse
    # pre = substring that precedes piece of interest
    # post = substring that follows piece of interest
    # returns piece of interest

    re_key = pre + "(.*)"+ post
    poi_search = re.search(re_key, seq)
    if poi_search is None:
        poi = "X"
    else:
        poi = poi_search.group(1)
    
    return poi

#putative consensus sequences ***reverse complement of snapgene***
rpp = 'AGCGGCC' #7bp ; before rptr barcode in read1
rpf = 'CTCGAGT' #7 bp ; after rptr barcode in read1

# function to parse RPTR fastq
def rbc_finder(readfile, bc_pre=rpp, bc_post=rpf, bc_len=14, chunk_size=1000):
    #readfile = fastq file -- RPTR files are read1
    #default values for the pre/post regions defined in above cell
    
    # Initialize DataFrame
    BC_df = pd.DataFrame(columns=["BCs", "Length", "Library"])

    # # Initialize Chunk Counter
    # chunk_count = 0

    # # Open file and read in chunks
    # seqlist = []
    # with open(readfile, 'r') as fin:
    #     while True:
    #         lines = []
    #         try:
    #             # Try to read chunk_size lines at a time
    #             for _ in range(chunk_size):
    #                 # print(next(fin))
    #                 lines.append(next(fin))
    #         except StopIteration:
    #             # If we've read all lines, break the loop
    #             if not lines:
    #                 break
    #         # print(lines)
    #         # Process lines
    #         for line in lines:
    #             if line.startswith('@'):
    #                 # Get the next line, which contains the read sequence
    #                 seq = next(fin, '').strip()  # Use next(fin, '') to avoid StopIteration here
    #                 if seq:
    #                     seqlist.append(seq)
    #         # print(seqlist)

    """TRYING OUT BIOPYTHON"""

    # Initialize an empty list to store sequences
    seqlist = []
    
    # Parse the FASTQ file and extract sequences
    for record in SeqIO.parse(readfile, "fastq"):
        seqlist.append(str(record.seq))  # Append the sequence as a string
    # print(seqlist)


    """END"""
    #make lists of BCs from list of reads
    bc_list = []
    bc_lens = []       
    for read in seqlist:
        bc = getmid(read, bc_pre, bc_post, bc_len)
        bc = reverse_complement(bc) #return reverse complement
        bc_list.append(bc)
        bcl = len(bc)
        bc_lens.append(bcl)

    #make the dict/df
    BC_dict = {"BCs":bc_list, "Length":bc_lens} 
    BC_df = pd.DataFrame.from_dict(BC_dict)
    
    #label df with library name
    libname = '_'.join(readfile.split('/')[-1].split('_')[0:6])
    BC_df['Library'] = libname
    
    return BC_df

# function that queries TBB table with a RPTRbc, returns the corresponding TBB, and inserts that TBB result into an empty list
def RPTR_SQLsearch(query):
    try:
        myquery = conn.sql("SELECT AD, AD_BC FROM A10_2_T_NODBLMAP_20240422 WHERE RPTR_BC='{}'".format(query)).to_df() #search for corresponding Tile-ADbc
        myquery ['AD_ADBC'] = myquery['AD'] + '-' +myquery['AD_BC'] # create a column that puts together ADs+BCs
        ptb = list(set(myquery ['AD_ADBC'].to_list())) #make a list out of that column and remove duplicates
        
        if not ptb:
            ptb = 0
        else:
            pass
#         print(len(ptb))

    except KeyError:
        ptb = 0

    return ptb #return the list of potential ADBCs 


#function to analyze reads
def SQLanalyze_tiles_rptrbcs (df, bc_len=14):
    # df = barcode containing df, parsed from fastq
    # bc_len = int, expected barcode length
    # tbb_dictkey = str, either 'ADbc' or 'RPTRbc'
    
    print(df.loc[0,'Library'])

    tr = df.shape[0] #total reads
    print(f'Total Reads {tr}')
    
    cls = df[(df['Length']== bc_len)] #cl = correct length
    print('Reads w BC and Tile of correct length')
    clcount = cls.shape[0]
    print(clcount)
    
    print('% Reads w correct length BCs')
    clpct = cls.shape[0]/df.shape[0]
    print (clpct)
    
    #df of BC coverage
    cl_covdf = cls['BCs'].value_counts().to_frame().reset_index() 
    print('# Unique RPTR BCs')
    uniqbccount = cl_covdf.shape[0]
    print(uniqbccount)
    print('SUM Unique BCs')
    print(cl_covdf.sum(numeric_only=True)['count'])

    #copy down unique BCs into a list
    bcs = cl_covdf['BCs'].tolist()
    


    matchlist = [] #list of TBBs that match the BCs

    # with concurrent.futures.ProcessPoolExecutor() as executor:
    #     results = executor.map(RPTR_SQLsearch, bcs)
    #     for r in results:
    #         matchlist.append(r)

    for x in bcs:
        myquery = RPTR_SQLsearch(x)
        matchlist.append(myquery)

            
    cl_covdf['PutativeTileADBC'] = matchlist
    
    matchesonly = cl_covdf.replace(0, np.nan)
    matchesonly = matchesonly.dropna()
    totmatches = matchesonly.sum(numeric_only=True)['count']
    print('# BC matches to A10 deep seq map (Thresholded)')
    mapmatches = matchesonly.shape[0]
    print(mapmatches)
    
    print('TOT BC matches to A10 deep seq map')
    print(totmatches)
    
#     print('% unique BCs matched')
#     print(totmatches/cl_covdf.shape[0])
    
    
    print()
    
    #label the df with library name
    
    libname = df.loc[0,'Library']
    matchesonly['Library'] = libname
    
    statslist = [libname, tr, clcount, clpct, uniqbccount, mapmatches, totmatches]
    print(statslist)
    
    return matchesonly

In [5]:
minifastq = 'MINI_4_10_RPTR.fastq'
minimap = rbc_finder(minifastq)
minimap

,BCs,Length,Library
0,AACCTAATCTCTTA,14,MINI_4_10_RPTR.fastq
1,ACCCCGAATGTTCT,14,MINI_4_10_RPTR.fastq
2,X,1,MINI_4_10_RPTR.fastq
3,TACGCGATTGATGC,14,MINI_4_10_RPTR.fastq
4,ACACCTTGCGCCGA,14,MINI_4_10_RPTR.fastq


In [6]:
rawmap_full_4_10_iii = rbc_finder(full_4_10_iii_path)
rawmap_full_4_10_iii

,BCs,Length,Library
0,TGGGTTGGTTGCCA,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
1,CGACGCGAGTACGT,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
2,GCAAGAGCAACAA,13,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
3,GGCCTAAGGCGAGC,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
4,X,1,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
...,...,...,...
23802971,TGAATCACACTTAC,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
23802972,CCGCGCTCCAATGA,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
23802973,TCTACACCTTGGCG,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
23802974,TATAGTAATAGTGT,14,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq


 23,802,976 in the pear file, 23802976 rows here YES

In [7]:
countsdf_full_4_10_iii = SQLanalyze_tiles_rptrbcs(rawmap_full_4_10_iii)
countsdf_full_4_10_iii

RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
Total Reads 23802976
Reads w BC and Tile of correct length
20666919
% Reads w correct length BCs
0.868249373523714
# Unique RPTR BCs
183935
SUM Unique BCs
20666919
# BC matches to A10 deep seq map (Thresholded)
15892
TOT BC matches to A10 deep seq map
15761143

['RPTR_4_10_S38_L001.fastq.gz.assembled.fastq', 23802976, 20666919, 0.868249373523714, 183935, 15892, 15761143]


,BCs,count,PutativeTileADBC,Library
0,CATTCATAAAGAAC,72087,[TTCTTCTCTTCTTCTGTTGATTCTACTCCAATGTTTGATTTGGAT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
1,GGAGGGCCAACGCA,61404,[TCTCCAGATATTGATGCTTCTCCATTTATTAATGATTCATTTGAA...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
2,ATGTCCATGACCTA,58751,[CCATCTATCTTTGATGGTTCTCCAGACTTTGATACATTTGATATT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
3,TAGTCAGACGCAAA,56118,[TTTACTGATTTGTCTACTCCATCATTTGATTCTCCAGGTTACTTC...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
4,TATCCCATTCCGAC,54892,[CCATCTATCTTTGATTCTCCAGATGTTGCTGAATCATTTGAAACT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
...,...,...,...,...
181233,CGATATCCCTGCAT,1,[GATTATTCTGGTTTGCAATCTGATTATTCTCCATTGACTGGTGTT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
181449,TGTGACATCGCTAT,1,[TCTTTGCCACCATTGCCACCAACTCCAAGATCTCCAGCTGTTGCT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
181729,CCGAGCGGTCTACT,1,[TCTGAATCTGTTGGTGAAGCTGTTAAATTGTTTAAACAATTGCCA...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq
181751,AGACGATCTCACTG,1,[AGACCAGAAGCTTTGCATAGACCTAAAGCTGCTTGTACTATTTCT...,RPTR_4_10_S38_L001.fastq.gz.assembled.fastq


In [48]:
rawmap = rbc_finder(test_4_10_path)
rawmap[rawmap['Length']==14].shape[0]/rawmap.shape[0]

0.8547427187022699

In [49]:
rawmap.shape[0]

18987

In [46]:
rawmapii = rbc_finder(test_4_10_ii_path)
rawmapii.dropna().shape[0] / rawmapii.dropna().shape[0]

12500

In [19]:
confirmedhas_testseq = 'TACATAACTAATTACATGAGCGGCCAGAACATTCGGGGTCTCGAGTTATTTAGAAGTTTATTTGTACAATTCATCCATACCATGGGTAATACCAGCAGCAGTAACAAATTCTAACAAGACCATGTGGTCTCTCTTTTCGTTTGGATCTTTGGATAAGGCAGATTGAGTGGATAAGTAATGGTTGTCTGGTAACAAGACTGGACCATCACCA'
getmid(confirmedhas_testseq, pre=rpp, post=rpf, bclen=14 )

'AGAACATTCGGGGT'

In [22]:
# peardata = 'pear_results.txt'
# pdata = []
# with open(peardata, 'r') as f: 
#     for line in f:
#         line = line.strip()
#         rawstats = line.split(':')
#         tot_paired = rawstats[1].split('/')[0]
#         tot_paired = int(tot_paired.replace(",", ""))
#         # print(tot_paired)
#         pdata.append(tot_paired)


pdata_files = 'pear_results_filepaths.txt'
fdata = []
with open(pdata_files, 'r') as f: 
    for line in f:
        line = line.strip()
        rawstats = line.split(':')
        rawfnames = rawstats[1].split('/')[1]
        libnames = rawfnames.split('_')[:6]
        print()
        # tot_paired = rawstats[1].split('/')[0]
        # tot_paired = int(tot_paired.replace(",", ""))
        # # print(tot_paired)
        # pdata.append(tot_paired)
# 	Library
# 1	Staller_AD_2_0_MVS_0057


['Staller', 'RPTR', '2', '15', 'MVS', '0081']
['Staller', 'AD', '2', '30', 'MVS', '0061']
['Staller', 'RPTR', '3', '0', 'MVS', '0085']
['Staller', 'AD', '4', '30', 'MVS', '0075']
['Staller', 'RPTR', '2', '10', 'MVS', '0080']
['Staller', 'RPTR', '2', '5', 'MVS', '0079']
['Staller', 'AD', '4', '10', 'MVS', '0073']
['Staller', 'AD', '3', '5', 'MVS', '0065']
['Staller', 'RPTR', '4', '15', 'MVS', '0095']
['Staller', 'RPTR', '4', '30', 'MVS', '0096']
['Staller', 'RPTR', '2', '30', 'MVS', '0082']
['Staller', 'RPTR', '3', '240', 'MVS', '0091']
['Staller', 'AD', '2', '5', 'MVS', '0058']
['Staller', 'RPTR', '3', '15', 'MVS', '0088']
['Staller', 'RPTR', '4', '240', 'MVS', '0050']
['Staller', 'AD', '2', '180', 'MVS', '0062']
['Staller', 'AD', '3', '240', 'MVS', '0070']
['Staller', 'AD', '3', '0', 'MVS', '0064']
['Staller', 'AD', '3', '15', 'MVS', '0067']
['Staller', 'AD', '4', '240', 'MVS', '0077']
['Staller', 'AD', '2', '240', 'MVS', '0063']
['Staller', 'RPTR', '4', '5', 'MVS', '0093']
['Staller'